All variables

In [ ]:
import xarray as xr
import os
from glob import glob

# =====================================================
# CONFIGURATION
# =====================================================

INPUT_DIR = r"Data\raster\EA_Selected_Farm_Data"
OUTPUT_DIR = r"Data\raster\rounded_climate"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =====================================================
# PROCESS FILES
# =====================================================

nc_files = glob(os.path.join(INPUT_DIR, "*.nc"))

print(f"Found {len(nc_files)} NetCDF files")

for nc_file in nc_files:
    filename = os.path.basename(nc_file)

    # -------------------------------------------------
    # Extract year from 'cYYYY'
    # -------------------------------------------------
    parts = filename.split(".")
    c_part = next(p for p in parts if p.startswith("c") and p[1:].isdigit())
    year = c_part[1:]   # remove 'c'

    # -------------------------------------------------
    # New filename
    # -------------------------------------------------
    new_filename = f"wheat_farm_{year}.nc"
    out_path = os.path.join(OUTPUT_DIR, new_filename)

    print(f"🔄 {filename} → {new_filename}")

    # -------------------------------------------------
    # Open, process (optional), save
    # -------------------------------------------------
    ds = xr.open_dataset(nc_file)

    # (Optional) round coords again if needed
    ds = ds.assign_coords(
        lat=ds.lat.round(2),
        lon=ds.lon.round(2)
    )

    ds.to_netcdf(out_path)
    ds.close()

    print(f"✅ Saved → {out_path}")

print("🎉 All files renamed and saved successfully!")


Target Variable

In [ ]:
import xarray as xr
import os
from glob import glob

# =====================================================
# CONFIGURATION
# =====================================================

INPUT_DIR = r"Data\raster\EA_Selected_Farm_Data"
OUTPUT_DIR = r"Data\raster\rounded_climate"

TARGET_VAR = "H_wheat_dot_hat"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =====================================================
# PROCESS FILES
# =====================================================

nc_files = glob(os.path.join(INPUT_DIR, "*.nc"))

print(f"Found {len(nc_files)} NetCDF files")

for nc_file in nc_files:
    filename = os.path.basename(nc_file)

    # -------------------------------------------------
    # Extract year from 'cYYYY'
    # -------------------------------------------------
    parts = filename.split(".")
    c_part = next(
        p for p in parts if p.startswith("c") and p[1:].isdigit()
    )
    year = c_part[1:]   # remove 'c'

    new_filename = f"wheat_farm_{year}.nc"
    out_path = os.path.join(OUTPUT_DIR, new_filename)

    print(f"🔄 {filename} → {new_filename}")

    # -------------------------------------------------
    # Open dataset
    # -------------------------------------------------
    ds = xr.open_dataset(nc_file)

    # -------------------------------------------------
    # Safety check
    # -------------------------------------------------
    if TARGET_VAR not in ds:
        print(f"⚠️ Skipping {filename} (missing {TARGET_VAR})")
        ds.close()
        continue

    # -------------------------------------------------
    # Keep ONLY wheat variable
    # -------------------------------------------------
    wheat_ds = ds[[TARGET_VAR]]

    # Round coordinates (recommended)
    wheat_ds = wheat_ds.assign_coords(
        lat=wheat_ds.lat.round(2),
        lon=wheat_ds.lon.round(2)
    )

    # -------------------------------------------------
    # Save cleaned NetCDF
    # -------------------------------------------------
    wheat_ds.to_netcdf(out_path)

    ds.close()
    wheat_ds.close()

    print(f"✅ Saved → {out_path}")

print("🎉 All wheat-only NetCDF files created successfully!")
